# 48. Mac에서 원판·자세 ONNX의 스레드 수와 지연 시간

같은 c04 116장을 매번 처음부터 읽습니다. 46·47번에서 확인한 FP32 고정형 그래프와 배치 1을 사용합니다. 두 모델에 같은 입력 텐서를 전달하고 CPU 연산 스레드만 2·4·6개로 바꿉니다. 각 설정을 2회 측정하며 순서는 4·2·6·6·2·4입니다. 매회 첫 프레임으로 10회 예열합니다.

영상 한 장을 읽기 시작할 때부터 두 모델의 NMS·원본 좌표 복원이 끝날 때까지가 `total`입니다. 파일 열기, 세션 초기화, 원판 궤적 연결, 그림 그리기, 인코딩·파일 저장은 제외합니다. 같은 프레임을 반복 측정한 수치입니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
from wlcv.evaluation.onnx_latency import benchmark, STAGES
import matplotlib.pyplot as plt

# 46·47번을 다시 실행했다면 아래 실행 폴더를 해당 결과로 바꾸세요.
out = root / "outputs"
plate = out / "onnx_plate_parity/20261125-194508-519898/plate.onnx"
pose = out / "onnx_pose_parity/20261125-200823-826057/pose.onnx"
folder = out / "onnx_cpu_latency" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
frames = benchmark(plate, pose, folder)
print(folder)

## 어떤 단계에 시간이 걸렸나?

단위는 프레임당 ms입니다. p50은 중앙값, p95는 프레임의 95%가 그 시간 안에 처리됐다는 뜻입니다. total은 각 프레임에서 직접 측정한 값입니다.

In [ ]:
steps = frames.groupby("threads")[STAGES].median()
steps["total_p95"] = frames.groupby("threads").total.quantile(.95)
display(steps.round(2))
display(frames.groupby(["trial", "threads"]).total.quantile([.50, .95]).unstack().round(2))
ax = steps[["decode", "prepare", "plate", "pose", "post"]].T.plot.bar(figsize=(8, 3.6), rot=0)
ax.set(ylabel="Median ms / frame", xlabel="Measured stage")
ax.legend(title="ORT CPU threads")
ax.figure.tight_layout()
ax.figure.savefig(folder / "stages.png", dpi=130)
plt.show()

## 같은 결과를 얻었나?

첫 4스레드 실행과 나머지 실행에서 같은 프레임의 원판·사람 후보 수, 직접 원판 기준과 관절 표시 기준을 대조합니다. 후보 순서대로 박스 좌표(최대 축 차이), 관절 좌표(두 점 사이 거리), 점수도 비교합니다. 후보 수가 다르면 좌표 오차가 비어 있고 상태 일치는 실패로 표시됩니다.

In [ ]:
checks = frames.groupby("threads").agg(
    same_counts_and_states=("same_counts_and_states", "all"),
    max_coordinate_delta_px=("coordinate_delta_px", "max"), max_score_delta=("score_delta", "max"))
display(checks)
steps.to_csv(folder / "latency.csv")
checks.to_csv(folder / "checks.csv")